# 第 5 章 推測: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/econometrics-programming/05-inference.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
seed = 0             # 乱数の種
n = 30               # 標本の大きさ
level = 0.95         # 信頼水準
repetitions = 100    # 標本を取り直す回数（区間の本数）
null_slope = 0.0     # 検定する帰無仮説の傾き（真の傾きは 0.5）

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

## 1 つの標本の t 値と信頼区間、被覆率と棄却率

In [ ]:
import numpy as np

from lib.chapter02.simple import draw_sample
from lib.chapter03.multiple import fit_ols
from lib.chapter05.inference import (
    confidence_interval,
    coverage_rate,
    rejection_rate,
    slope_intervals,
    t_statistics,
)

rng = np.random.default_rng(seed)
x, y = draw_sample(n, 1.0, 0.5, 1.0, rng)
result = fit_ols(y, x)
t = t_statistics(result, null_slope).iloc[1]
print(f"傾き {result.coefficients.iloc[1]:.3f}、t 値（帰無仮説 {null_slope}） {t:.3f}")
print(confidence_interval(result, level).round(3))

intervals = slope_intervals(n, repetitions, level, np.random.default_rng(seed))
contains = ((intervals["lower"] <= 0.5) & (0.5 <= intervals["upper"])).mean()
print(f"{repetitions} 本のうち、真の傾きを含む割合 {contains:.2f}")
print(f"被覆率（2000 回） "
      f"{coverage_rate(n, 2000, level, np.random.default_rng(seed)):.3f}")
rate = rejection_rate(n, 1000, 0.5, null_slope, 0.05, np.random.default_rng(seed))
print(f"棄却率（真の傾き 0.5、帰無仮説 {null_slope}、有意水準 5%） {rate:.3f}")

## 図: 標本ごとの信頼区間

In [ ]:
from lib.chapter05.plot import plot_confidence_intervals

plot_confidence_intervals(intervals, 0.5)

## 試してみよう

- `level` を 0.8 にすると、区間は狭くなり、真の傾きを含む割合も、0.97 から 0.78
  に下がります（被覆率は 0.812。水準に近い値です）
- `n` を 200 にすると、1 つの標本の t 値は 0.561 から 5.694 になり、棄却率は 0.703
  から 1.000 になります（標本が大きいほど、真の傾き 0.5 が、0 と違うと見抜けます）
- `null_slope` を 0.5（真の値）にすると、棄却率は 0.047 になり、有意水準 5%
  に近づきます。正しい帰無仮説を、誤って棄却する割合です